Environment Setup & Dataset Download: 
This cell installs/imports dependencies, downloads the tiny Shakespeare dataset, and loads it into memory.

In [2]:
pip install torch

  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/126.7 MB ? eta -:--:--
   ---------------------------------------- 1.3/126.7 MB 8.4 MB/s eta 0:00:15
    --------------------------------------- 2.4/126.7 MB 6.4 MB/s eta 0:00:20
   - -------------------------------------- 3.7/126.7 MB 5.7 MB/s eta 0:00:22
   - -------------------------------------- 4.7/126.7 MB 5.6 MB/s eta 0:00:22
   - -------------------------------------- 5.8/126.7 MB 5.3 MB/s eta 0:00:23
   -- ------------------------------------- 6.8/126.7 MB 5.3 MB/s eta 0:00:23
   -- ------------------------------------- 7.9/126.7 MB 5.2 MB/s eta 0:00:23
   -- ------------------------------------- 9.2/126.7 MB 5.3 MB/s eta 0:00:23
   --- ------------------------------------ 10.2/126.7 MB 5.3 MB/s eta 0:00:22
   --- ------------------------------------ 11.3/126.7 MB 5.3 MB/s eta 0:00:22
   --- ------------------------------------ 12.3/126.7 MB 5.2 MB/s eta 0:00:22


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
import urllib.request
import torch
import torch.nn as nn
from torch.nn import functional as F

# Download a small public-domain dataset (Tiny Shakespeare sample)
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
dataset_path = "dataset.txt"
urllib.request.urlretrieve(url, dataset_path)

# Load and inspect the dataset
with open(dataset_path, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Dataset loaded successfully!")
print(f"Total characters in dataset: {len(text):,}")
print("\n--- First 300 characters ---")
print(text[:300])

Dataset loaded successfully!
Total characters in dataset: 1,115,394

--- First 300 characters ---
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


Tokenization & Vocabulary Creation: 
This cell builds the character vocabulary and implements the encoding/decoding functions, then converts the dataset into a PyTorch tensor.

In [4]:
# Build character vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocabulary size (unique characters): {vocab_size}")
print(f"Characters: {''.join(chars)}")

# Create mapping from characters to integers and vice versa
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s] # encoder: string -> list of ints
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: list of ints -> string

# Encode the entire text dataset and store it into a PyTorch tensor
data = torch.tensor(encode(text), dtype=torch.long)
print(f"\nData tensor shape: {data.shape}")

Vocabulary size (unique characters): 65
Characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz

Data tensor shape: torch.Size([1115394])


Training/Validation Split & Batching Setup: 
This cell splits the data into training and validation sets and defines helper functions to sample batches and estimate losses.

In [5]:
# Hyperparameters
batch_size = 32
block_size = 8
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1337)

# Create separate training and validation data (90% train, 10% val)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    # Generate a small batch of data of inputs x and targets y
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

print(f"Using device: {device}")
xb, yb = get_batch('train')
print("inputs x shape:", xb.shape)
print("targets y shape:", yb.shape)

Using device: cpu
inputs x shape: torch.Size([32, 8])
targets y shape: torch.Size([32, 8])


Define the Bigram Language Model Architecture: 
This cell defines the BigramLanguageModel class using PyTorch embedding tables and a text generation method.

In [6]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Each token reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B, T) tensor of integers
        logits = self.token_embedding_table(idx) # (Batch, Time, Vocab_size)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # Get the predictions
            logits, loss = self(idx)
            # Focus only on the last time step
            logits = logits[:, -1, :] # Becomes (B, C)
            # Apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # Append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel(vocab_size)
m = model.to(device)

Generate Sample Before Training: 
This cell generates text using the untrained model to establish a baseline.

In [7]:
print("--- GENERATED SAMPLE BEFORE TRAINING ---")
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_before = decode(m.generate(context, max_new_tokens=200)[0].tolist())
print(generated_before)

--- GENERATED SAMPLE BEFORE TRAINING ---

JLg,3D&OM .3YCjfolRwqXaDyttW!GmaUT-IIvuZV?sYfjzUvTQ3RwL ?etyLeg.COHW
Ri$ELkJMXpBEX;-G&Orl!bcH ;cq.z,rbyQZoN:QVT:cVOtkTLhuMYe-gqGhTxDlqYBkLDJnAsLJOVeJYD
J:r:HHESAbIqa!SKO.zJkSD$AzSQemsLuMElSU
NjOaXnHFz


Train the Bigram Language Model: 
This cell runs the training loop, periodically evaluating training and validation loss.

In [8]:
# Training hyperparameters
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
eval_iters = 200

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

print("Starting Training Loop...")
for iter in range(max_iters):
    # Every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # Sample a batch of data
    xb, yb = get_batch('train')

    # Evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("Training completed!")

Starting Training Loop...
step 0: train loss 4.7725, val loss 4.7793
step 300: train loss 2.8278, val loss 2.8480
step 600: train loss 2.5552, val loss 2.5647
step 900: train loss 2.5083, val loss 2.5242
step 1200: train loss 2.4806, val loss 2.5059
step 1500: train loss 2.4754, val loss 2.4997
step 1800: train loss 2.4703, val loss 2.4869
step 2100: train loss 2.4624, val loss 2.4945
step 2400: train loss 2.4581, val loss 2.4945
step 2700: train loss 2.4686, val loss 2.4875
step 2999: train loss 2.4722, val loss 2.4855
Training completed!


Generate Sample After Training & Compare: 
This cell generates text after training so you can compare the output against the untrained baseline.

In [9]:
print("--- GENERATED SAMPLE AFTER TRAINING ---")
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_after = decode(m.generate(context, max_new_tokens=200)[0].tolist())
print(generated_after)

--- GENERATED SAMPLE AFTER TRAINING ---

Wanelo ol t hes t te
HarenthrvosenindizGouncesere ballerrang;
VINI ho he t mare.
Tuc;
Houd wnad; foured asiny we letund;
Ak beZARo g ls we MII:
ARUCHemese's we ant westh afre CI ond.
Dqup fof pt wet d
